In [ ]:
!pip install -q open_clip_torch num2words

In [ ]:
# imports

import os
import io
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
import open_clip

In [ ]:
# configuration

DATASET_ID     = "UBC-SLIME/colx585_group_project_data"
CLIP_MODEL     = "ViT-B-32"
CLIP_PRETRAINED = "openai"

CHECKPOINT_DIR = "./clip_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_EVERY = 500

In [ ]:
# load CLIP model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
clip_model     = clip_model.to(device).eval()
print("CLIP model loaded.")

In [ ]:
# helper functions

def load_image(example):
    if isinstance(example["image"], dict) and "bytes" in example["image"]:
        return Image.open(io.BytesIO(example["image"]["bytes"])).convert("RGB")
    return example["image"].convert("RGB")


@torch.no_grad()
def clip_alignment_score(example):
    """
    Compute CLIP cosine similarity between image and caption.

    Members (seen_both) are genuine image-caption pairs → high alignment.
    Non-members include at least one mismatched component → lower alignment.
    """
    image   = load_image(example)
    caption = (example["text"].split("\n", 1)[1]
               if "\n" in example["text"]
               else example["text"])

    # Encode image
    image_tensor   = clip_preprocess(image).unsqueeze(0).to(device)
    image_features = clip_model.encode_image(image_tensor)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    # Encode caption (CLIP tokenizer max length is 77)
    text_tokens   = clip_tokenizer([caption[:77]]).to(device)
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Cosine similarity
    similarity = (image_features * text_features).sum(dim=-1)
    return float(similarity.item())

In [ ]:
# evaluation helpers

def tpr_at_fpr(y_true, y_score, target_fpr=0.1):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    valid = np.where(fpr <= target_fpr)[0]
    return float(np.max(tpr[valid])) if len(valid) > 0 else 0.0


def evaluate(df, split_name):
    if "label" not in df.columns or df["label"].isna().all():
        print(f"{split_name}: no labels available (test split).")
        return
    auc   = roc_auc_score(df["label"], df["score"])
    tpr01 = tpr_at_fpr(df["label"].values, df["score"].values)
    print(f"[{split_name}]  AUC = {auc:.4f}  |  TPR@FPR=0.1 = {tpr01:.4f}")

In [ ]:
# scoring loop

def score_split(split_name):
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{split_name}_scores.csv")

    completed_ids = set()
    records = []
    if os.path.exists(ckpt_path):
        ckpt_df = pd.read_csv(ckpt_path)
        records = ckpt_df.to_dict("records")
        completed_ids = set(ckpt_df["id"].tolist())
        print(f"Resuming from checkpoint: {len(completed_ids)} samples already scored.")

    ds = load_dataset(DATASET_ID, split=split_name)

    for i, example in enumerate(tqdm(ds, desc=f"CLIP scoring {split_name}")):
        sample_id = example["id"]
        if sample_id in completed_ids:
            continue

        try:
            score = clip_alignment_score(example)
            record = {"id": sample_id, "score": score}
            if example.get("is_member") is not None:
                record["label"] = example["is_member"]
            if example.get("type") is not None:
                record["type"] = example["type"]
            records.append(record)

        except Exception as e:
            print(f"  Error on sample {sample_id}: {e} — skipping.")
            records.append({"id": sample_id, "score": 0.0})

        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(ckpt_path, index=False)
            print(f"  Checkpoint saved at sample {i + 1}.")

    df = pd.DataFrame(records)
    df.to_csv(ckpt_path, index=False)
    print(f"Scoring complete. Results saved to {ckpt_path}")
    return df

In [ ]:
# run validation

df_val = score_split("validation")
evaluate(df_val, "validation")

# Per-type breakdown for report
if "label" in df_val.columns and "type" in df_val.columns:
    print("\nPer-type AUC breakdown:")
    for t, group in df_val.groupby("type"):
        if group["label"].nunique() > 1:
            auc = roc_auc_score(group["label"], group["score"])
            print(f"  {t:<30}  AUC = {auc:.4f}  (n={len(group)})")

    print("\nMean CLIP score per type:")
    for t, group in df_val.groupby("type"):
        print(f"  {t:<30}  mean = {group['score'].mean():.4f}  (n={len(group)})")

In [ ]:
# run test and submit

df_test = score_split("test")

# Save Kaggle submission
output_path = "submission_clip_test.csv"
submission  = df_test[["id", "score"]].rename(columns={"score": "is_member"})
submission.to_csv(output_path, index=False)
print(f"\nKaggle submission saved: {output_path}  ({len(submission)} rows)")